# 两版冻结配置模型的开盘路由消融
固定6个新账户，唯一干预为已有model=None路径每天执行风险投影后的目标；保留全部风险和恢复规则。不拟合、不挑成本、不扫阈值。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/open_router_ablation_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
names=['inputs/plan','analysis/targets',*[f'analysis/native/{family}_{cost}' for family in ['candidate','soxx'] for cost in plan['slippage_bps']]]
strong,strong_audit=_source(root,plan['frozen_source'],names);opening,opening_audit=_source(root,plan['opening_model_source'],names)
source,source_audit=_source(root,plan['source'],['inputs/frame']);controller,controller_audit=_source(root,plan['price_source'],['inputs/controller','inputs/plan']);action,action_audit=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection'])
for existing in [strong,opening]:
 for key in ['tariff','initial_cash','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert existing['inputs/plan'][key]==plan[key]
assert controller['inputs/plan']['tariff']==plan['tariff']
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec'])
modified=dict(controller['inputs/controller']);modified['frozen_open_action_value_model']=None
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(plan['start'],plan['end'])].copy()
books={};prefix=[]
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'strong':strong_audit,'opening':opening_audit,'market':source_audit,'controller':controller_audit,'action':action_audit});rec.save_objects(artifact_path='sources',routing=(root/'src/workflow/stage1_open_action_value.py').read_text(),replay=(root/'src/backtest/full_path_tiered.py').read_text());rec.log_params(maximum_new_native_cases=6,new_fits=0)
for name,original in [('opening',opening),('strong',strong)]:
 data=frame.copy();targets=original['analysis/targets']['candidate'].reindex(pd.DatetimeIndex(frame.signal_date));assert targets.notna().all().all()
 for col in targets:data[col]=targets[col].to_numpy()
 for cost in plan['slippage_bps']:
  replay=replay_full_path(data,models=None,plan=plan,slippage_bps=cost,controller=modified,recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection'])
  assert replay['replay'].execute_open_target.all()
  book=native_tiered_stage1(data,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=cost,initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition)
  control=original[f'analysis/native/candidate_{cost}'];books[(name+'_always',cost)]=book;books[(name+'_router',cost)]=control;books[('soxx',cost)]=strong[f'analysis/native/soxx_{cost}']
  assert book['authority'].trade_date.equals(control['authority'].trade_date)
  first=np.flatnonzero(~control['authority'].execute_open_target.to_numpy());stop=int(first[0]) if len(first) else len(frame)
  cols=['equity','ending_soxl_quantity','ending_soxx_quantity','ending_cash'];np.testing.assert_allclose(book['authority'][cols].iloc[:stop],control['authority'][cols].iloc[:stop],atol=1e-8,rtol=0)
  prefix.append({'family':name,'slippage_bps':cost,'unchanged_prefix_sessions':stop,'first_original_hold':str(frame.trade_date.iloc[stop].date()) if stop<len(frame) else None})
  rec.save_objects(artifact_path='analysis/native',**{f'{name}_always_{cost}':book,f'{name}_router_{cost}':control});rec.log_metrics(completed_new_native_cases=len(prefix))
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'orders':len(book['orders']),'broker_fees_usd':path.broker_and_external_fees_usd.sum(),'slippage_usd':path.slippage_reserve_usd.sum(),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()})
 annual.append(annual_paired_metrics(path,bench).assign(family=family,slippage_bps=cost))
 if family.endswith('_always'):
  for comparator in ['soxx',family.replace('_always','_router')]:uncertainty.append({'family':family,'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);gates={}
for family in ['opening','strong']:
 cand=table[table.family.eq(family+'_always')].set_index('slippage_bps');control=table[table.family.eq(family+'_router')].set_index('slippage_bps')
 gates[family]={'beat_paired_all_costs':bool(cand.terminal_wealth.gt(control.terminal_wealth).all()),'net_soxx_excess_all_costs':bool(cand.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cand.max_drawdown_pct.ge(-30).all() and cand.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cand.cvar5_pct.ge(cand.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cand.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq(family+'_always')].groupby('slippage_bps').relative_wealth.apply(lambda v:v.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty[uncertainty.family.eq(family+'_always')].lower_bound_positive.all()),'independent_development':False,'qualified_forward':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/native/*','inputs/*','sources/*']}
summary={'experiment':card.name,'decision':'not_promoted_fixed_open_router_ablation','gates':gates,'new_fits':0,'new_native_cases':6,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':plan['scope']}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,prefix=pd.DataFrame(prefix),summary=summary);rec.log_metrics(registered_artifact_count=21)
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('prefix_verification',pd.DataFrame(prefix))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/opening_state_training_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# 开盘执行路由消融：长期模型历史改善，但仍无稳定SOXX超额

完成0拟合、6个新增完整Qlib原生账户。固定两套冻结配置：近期执行标签+开盘特征模型，以及长期训练小配置头。唯一干预为已有路由model=None分支，即每天执行经过原风险投影的目标；停止/等待/恢复状态、12%与22%规则、近期损失上限、恢复模型、SOXL30%上限、现金15%底线、EMA及目标保持逻辑均沿用。账户风险状态随各自净值演化，没有跨成本冻结路径。生产release未修改。

本金10万美元、同一IBKR Pro Tiered费用投影，0/2/5bp独立结算，2023-01-03信号至2026-10-09退出共944区间。干预发生前与原账户现金、股数、净值逐值核对通过：开盘模型各14日、长期模型各10日。全部新账户每日执行目标标记为真，并通过原生订单费用、现金、股数及净值对账。

| 模型与执行方式 | 0bp净收益 | 2bp净收益 | 5bp净收益 | 2bp监控最大回撤 |
|---|---:|---:|---:|---:|
| 开盘模型原路由 | +102.24% | +139.55% | +118.44% | 31.21% |
| 开盘模型每天执行 | +91.26% | +85.41% | +121.66% | 30.20% |
| 长期模型原路由 | +342.52% | +335.36% | +348.06% | 27.64% |
| 长期模型每天执行 | +358.11% | +359.33% | +351.07% | 27.71% |
| SOXX | +387.75% | +387.73% | +387.70% | 43.39% |

长期模型每天执行在三个成本下胜过自身配对路由，监控回撤27.65%至27.80%，但净SOXX超额、年度/滚动70%和bootstrap正下界门均失败。126日正超额比例仅43.59%/45.91%/45.30%。2bp订单由1095笔增至1337笔，不是依靠减少交易费用改善；还存在状态路径反馈，不能把全部收益差称为分类器预测损失。开盘模型只在5bp小幅改善，0/2bp明显退步，三个成本监控回撤均超过30%。模型间方向差异不支持普遍取消路由。

本轮只有一项固定干预，未搜索执行概率阈值、交易规模、等待长度或风险参数；两模型提前指定，全部成本均报告。历史评价窗已反复复用，且未做独立开发/合格前向验证，不能称为稳定超额或可部署成果。监控回撤仍是既定时点口径，不是全分钟最高回撤；成交/派息为历史研究口径，非真实券商结算认证。

决策not_promoted_fixed_open_router_ablation。长期模型每天执行可保留为新的历史研究参考，但不替换生产，也不据此重开同一消融的阈值搜索。固定路线关闭。下一信息需求是新配置模型下严格时间外的execute-versus-HOLD净优势标签：若考虑重训路由，必须由训练期内部时间切分生成配置预测，再建立动作标签；不得拿本轮944日收益拟合或挑路由规则。尚未证明新路由可提供独立增量，稳定净SOXX超额目标仍未完成。

Qlib模型/原生账户证据及年度、滚动、bootstrap汇总随卡保存，NAS冷恢复逐对象哈希全部通过；仅使用已有执行分支，无新增生产逻辑或重复软件测试。
